# Prompting Techniques — Examples
## Zero-Shot · Few-Shot · Chain-of-Thought

| Technique | What it is |
|---|---|
| **Zero-shot** | Ask directly — no examples given |
| **Few-shot** | Show 2–3 examples before the question |
| **Chain-of-Thought (CoT)** | Force step-by-step reasoning before the answer |


In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
from openai import OpenAI

# OpenRouter uses the OpenAI-compatible API
# Set OPENROUTER_API_KEY in your environment variables
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

# Model to use throughout this course
# OpenRouter format: "provider/model-name"
MODEL = "anthropic/claude-sonnet-4-5"


In [ ]:
def get_completion(prompt, system="You are a helpful assistant.", temperature=0):
    """
    Simple wrapper around OpenRouter API.

    Args:
        prompt      : the user message
        system      : the system prompt
        temperature : 0 = deterministic, 1 = creative
    
    Returns:
        The model's response as a plain string.
    """
    response = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt},
        ]
    )
    return response.choices[0].message.content


---
## Part 1 — Zero-Shot Prompting

**What it is:** Ask the model a question with no examples — just the instruction and the input.

### Example 1 — Sentiment Classification

In [ ]:
system = """Your task is to classify the sentiment of the customer review.
Respond with only one word: positive, negative, or neutral."""

review = "The delivery was fast and the product works perfectly. Really happy with my purchase!"

response = get_completion(review, system=system)
print("Input:", review)
print("Output:", response)


### Example 2 — Named Entity Extraction (NER)

In [ ]:
system = """Extract named entities from the text.
Return a JSON list: [{"span": "entity text", "label": "PER/ORG/LOC/DATE"}]
If no entities found, return []."""

text = "Dr. Sarah Johnson from MIT announced Pfizer released a new drug in Boston last March."

response = get_completion(text, system=system)
print("Input:", text)
print()
print("Output:", response)


**When does zero-shot fail?**  
When you need a very specific output format or custom labels the model hasn't seen before.  
That's where few-shot helps.


---
## Part 2 — Few-Shot Prompting

**What it is:** Provide 2–3 examples before your actual question. The model learns your desired format from the examples.

### Example 1 — Consistent Answer Style

In [ ]:
def few_shot_style(topic):
    """Uses conversation history as few-shot examples to set answer style."""
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system",    "content": "Answer in a consistent style matching the examples."},
            {"role": "user",      "content": "Teach me about patience."},
            {"role": "assistant", "content": "Patience is the ability to wait calmly, knowing that the right outcome takes time."},
            {"role": "user",      "content": "Teach me about discipline."},
            {"role": "assistant", "content": "Discipline is training yourself to act consistently, even when motivation fades."},
            {"role": "user",      "content": f"Teach me about {topic}."},
        ]
    )
    return response.choices[0].message.content

for topic in ["courage", "curiosity", "empathy"]:
    print(f"Topic: {topic}")
    print(f"Answer: {few_shot_style(topic)}")
    print()


### Example 2 — Sentiment with Custom Labels

Few-shot teaches the model your custom label schema: `happy`, `frustrated`, `confused`.

In [ ]:
def classify_custom_sentiment(review):
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system",    "content": "Classify the review. Respond with one word: happy, frustrated, or confused."},
            {"role": "user",      "content": "Review: This is exactly what I needed, works perfectly!"},
            {"role": "assistant", "content": "happy"},
            {"role": "user",      "content": "Review: I have contacted support three times and nothing is resolved."},
            {"role": "assistant", "content": "frustrated"},
            {"role": "user",      "content": "Review: The manual says press A but there is no A button on my device?"},
            {"role": "assistant", "content": "confused"},
            {"role": "user",      "content": f"Review: {review}"},
        ]
    )
    return response.choices[0].message.content

reviews = [
    "I can't figure out how to set this up, the instructions make no sense.",
    "Absolutely love it! Best purchase I've made this year.",
    "I ordered blue but it arrived looking different from the website photo.",
]

for r in reviews:
    label = classify_custom_sentiment(r)
    print(f"Review: {r}")
    print(f"Label:  {label}")
    print()


---
## Part 3 — Chain-of-Thought (CoT) Prompting

**What it is:** Force the model to reason step by step before giving the final answer.

```
<thinking>  →  internal reasoning (hidden from users)
<response>  →  final answer (shown to users)
```

### Example 1 — Food Menu Chatbot

In [ ]:
food_items = """
Menu: Kids Menu    
  Mini Cheeseburger — $6.99 — beef patty, cheese, lettuce, tomato, fries

Menu: Appetizers
  Loaded Potato Skins — $8.99 — cheese, bacon, sour cream
  Bruschetta — $7.99 — tomato, basil, garlic, balsamic (VEGAN)

Menu: Main Menu
  Grilled Chicken Caesar Salad — $12.99
  Classic Cheese Pizza — $10.99 (MOST POPULAR)
  Spaghetti Bolognese — $14.99

Menu: Vegan Options
  Veggie Wrap — $9.99
  Vegan Beyond Burger — $11.99

Menu: Desserts
  Chocolate Lava Cake — $6.99
  Fresh Berry Parfait — $5.99 (VEGAN)
"""

system = f"""You are a restaurant chatbot.

Menu items:
<menu_items>
{food_items}
</menu_items>

Follow these steps for every question:
<thinking>
Step 1: Is this question about food? If not, politely decline.
Step 2: Is the item on the menu? If not, suggest similar items.
Step 3: Answer using only the menu data above.
</thinking>
<response>
Your answer to the customer here.
</response>
"""

def ask_menu_clean(question):
    """Returns only the clean response — hides the thinking block."""
    raw = get_completion(question, system=system)
    try:
        start = raw.index("<response>") + len("<response>")
        end   = raw.index("</response>")
        return raw[start:end].strip()
    except ValueError:
        return raw

questions = [
    "How much is the cheese pizza?",
    "Do you have any vegan desserts?",
    "What is the weather today?",
]

for q in questions:
    print(f"Customer: {q}")
    print(f"Bot:      {ask_menu_clean(q)}")
    print()


### Example 2 — NER with CoT

CoT forces the model to reason about entity boundaries before committing to the JSON output.

In [ ]:
import json

system = """You are an expert NER system.

Entity types: PER (person) · ORG (organisation) · LOC (location) · DATE (date/time) · MED (medical)

For every input, follow these steps:
<thinking>
Step 1 - READ: Identify all candidate entity spans.
Step 2 - CLASSIFY: For each span, decide the type and reason why.
Step 3 - VALIDATE: Check boundary accuracy and label correctness.
</thinking>

<response>
<entities>
[{"span": "entity text", "label": "TYPE", "confidence": "high/medium/low"}]
</entities>
</response>
"""

def extract_entities(text):
    raw = get_completion(text, system=system)

    # Extract thinking (for learning/debugging)
    thinking = ""
    if "<thinking>" in raw:
        thinking = raw[raw.index("<thinking>") + len("<thinking>"):raw.index("</thinking>")].strip()

    # Extract JSON
    entities = []
    if "<entities>" in raw:
        json_str = raw[raw.index("<entities>") + len("<entities>"):raw.index("</entities>")].strip()
        entities = json.loads(json_str)

    return entities, thinking

sentences = [
    "Elon Musk confirmed Tesla will open a new Gigafactory in Berlin by January 2025.",
    "The WHO reported that Moderna vaccine reduced COVID-19 cases in London last March.",
]

for sentence in sentences:
    entities, thinking = extract_entities(sentence)
    print(f"TEXT: {sentence}")
    print()
    print("REASONING:")
    print(thinking)
    print()
    print("ENTITIES:")
    for e in entities:
        print(f"  [{e['label']}] {e['span']:<30} confidence: {e['confidence']}")
    print("=" * 70)


---
## Summary

| Technique | When to use | Key advantage |
|---|---|---|
| **Zero-shot** | Simple, well-known tasks | Fast, no examples needed |
| **Few-shot** | Custom output format or labels | Teaches the model your style |
| **CoT** | Complex tasks needing reasoning | Fewer mistakes on ambiguous inputs |

```
Simple task?           → Zero-shot
Custom format/labels?  → Few-shot
Complex logic?         → Chain-of-Thought
```
